# Per-gate Quantum Dropout vs Depolarising Output Noise
**Reproducible Colab notebook** — resolves the open question of the WETICE paper:
does *per-gate stochastic deactivation* regularise where the depolarising channel fails?

`gap = train_acc − test_acc` (positive = overfitting). A true regulariser **reduces** the gap vs the noiseless VQC.

> Self-contained: downloads NSL-KDD (falls back to a synthetic XOR task if the download fails),
> no progress bars, small outputs — to avoid the notebook-JSON corruption seen with large/`tqdm` outputs.
> **Before saving: Edit → Clear all outputs.**

In [ ]:
# 1. Dependencies (quiet)
!pip install -q pennylane scikit-learn pandas matplotlib >/dev/null 2>&1
import numpy as np, pennylane as qml
from pennylane import numpy as pnp
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
print("pennylane", qml.__version__)

## 2. Data — NSL-KDD (real) with synthetic XOR fallback

In [ ]:
NSLKDD_COLS = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
 "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised","root_shell",
 "su_attempted","num_root","num_file_creations","num_shells","num_access_files","num_outbound_cmds",
 "is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate","rerror_rate",
 "srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
 "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
 "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate",
 "dst_host_srv_rerror_rate","label","difficulty"]

def load_data(n_samples=900, d=4, seed=7):
    """Return standardised PCA features (Xtr,Xte,ytr,yte) shared by all models."""
    try:
        import pandas as pd, urllib.request, os
        url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt"
        if not os.path.exists("nslkdd.txt"):
            urllib.request.urlretrieve(url, "nslkdd.txt")
        df = pd.read_csv("nslkdd.txt", header=None, names=NSLKDD_COLS)
        df = df.sample(n=min(n_samples, len(df)), random_state=seed).reset_index(drop=True)
        y = (df["label"].values != "normal").astype(int)
        X = df.drop(columns=["label","difficulty"])
        for c in ["protocol_type","service","flag"]:
            X[c] = LabelEncoder().fit_transform(X[c].astype(str))
        X = StandardScaler().fit_transform(X.astype(float).values)
        src = "NSL-KDD (real)"
    except Exception as e:                        # offline fallback: XOR-structured task
        rng = np.random.default_rng(seed)
        cen = np.array([[1,1],[-1,-1],[1,-1],[-1,1]],float); lab = np.array([0,0,1,1])
        idx = rng.integers(0,4,n_samples)
        X = cen[idx] + rng.normal(0,0.35,(n_samples,2)); y = lab[idx]
        X = np.c_[X, rng.normal(0,1,(n_samples,6))]   # 8 feats, only 2 informative
        X = StandardScaler().fit_transform(X); src = f"synthetic XOR (download failed: {type(e).__name__})"
    Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.3,random_state=seed,stratify=y)
    pca = PCA(n_components=d, random_state=seed).fit(Xtr)
    Xtr,Xte = pca.transform(Xtr), pca.transform(Xte)
    sc = StandardScaler().fit(Xtr)
    print(f"data: {src} | train {Xtr.shape} test {Xte.shape} | attack rate {yte.mean():.2f}")
    return sc.transform(Xtr), sc.transform(Xte), ytr, yte

Xtr_full, Xte, ytr_full, yte = load_data()

## 3. Ring SQNN with three modes: none / depolarising / per-gate dropout

In [ ]:
class QNNS_Dropout:
    def __init__(self, n_qubits=4, n_layers=3, p_deco=0.0, p_drop=0.0, seed=0):
        self.nq, self.nl = n_qubits, n_layers
        self.p_deco, self.p_drop = p_deco, p_drop
        self.dev = qml.device("default.mixed", wires=n_qubits)
        rng = np.random.default_rng(seed)
        self.params = {"w_in":  pnp.array(rng.normal(0,1.0,(n_layers,n_qubits)), requires_grad=True),
                       "w_rot": pnp.array(rng.normal(0,0.3,(n_layers,n_qubits,2)), requires_grad=True),
                       "a": pnp.array(2.0, requires_grad=True), "b": pnp.array(0.0, requires_grad=True)}
        self._rng = rng; self._build()
    def _build(self):
        @qml.qnode(self.dev, interface="autograd", diff_method="backprop")
        def qnode(x, w_in, w_rot, mask, p_deco):
            for l in range(self.nl):
                for i in range(self.nq):
                    feat = x[..., i % x.shape[-1]]
                    qml.RY(w_in[l,i]*feat + mask[l,i,0]*w_rot[l,i,0], wires=i)   # data kept, variational masked
                    qml.RZ(mask[l,i,1]*w_rot[l,i,1], wires=i)
                for i in range(self.nq): qml.CNOT(wires=[i,(i+1)%self.nq])       # ring
                if p_deco>0:
                    for i in range(self.nq): qml.DepolarizingChannel(p_deco, wires=i)
            return qml.expval(qml.PauliZ(0))
        self.qnode = qnode
    def _proba(self, X, p, mask):
        z = self.qnode(X, p["w_in"], p["w_rot"], mask, self.p_deco)
        return 1.0/(1.0+pnp.exp(-(p["a"]*z + p["b"])))
    def fit(self, X, y, epochs=60, lr=0.12):
        Xb = pnp.array(X, requires_grad=False); yb = pnp.array(y.astype(float), requires_grad=False)
        ones = pnp.array(np.ones((self.nl,self.nq,2)), requires_grad=False)
        opt = qml.AdamOptimizer(lr); keys = list(self.params.keys())
        def cost(*vals, mask=ones):
            p = dict(zip(keys, vals)); pr = pnp.clip(self._proba(Xb,p,mask),1e-7,1-1e-7)
            return -pnp.mean(yb*pnp.log(pr)+(1-yb)*pnp.log(1-pr))
        vals = [self.params[k] for k in keys]
        for _ in range(epochs):
            if self.p_drop>0:
                m = (self._rng.random((self.nl,self.nq,2)) > self.p_drop).astype(float)
                mask = pnp.array(m, requires_grad=False)
            else: mask = ones
            vals = opt.step(lambda *v: cost(*v, mask=mask), *vals)
        self.params = dict(zip(keys, vals)); self._ones = ones; return self
    def predict(self, X):
        pr = self._proba(pnp.array(X, requires_grad=False), self.params, self._ones)
        return (np.array(pr) >= 0.5).astype(int)

## 4. Experiment — train–test gap by mode and training size
*(a few minutes on a CPU runtime; reduce `seeds`/`sizes` to go faster)*

In [ ]:
import warnings; warnings.filterwarnings("ignore")
sizes = [40, 80, 160]; seeds = [1, 2]
conds = {"VQC (none)":  dict(p_deco=0.0,  p_drop=0.0),
         "Depol 0.05":  dict(p_deco=0.05, p_drop=0.0),
         "QDropout 0.3":dict(p_deco=0.0,  p_drop=0.3)}
results = {}
for n in sizes:
    for cname, cfg in conds.items():
        gaps = []
        for sd in seeds:
            idx = np.random.default_rng(1000+sd).choice(len(Xtr_full), size=n, replace=False)
            m = QNNS_Dropout(n_qubits=4, n_layers=3, seed=sd, **cfg).fit(Xtr_full[idx], ytr_full[idx], epochs=60)
            tr = accuracy_score(ytr_full[idx], m.predict(Xtr_full[idx]))
            te = accuracy_score(yte, m.predict(Xte))
            gaps.append(tr - te)
        results[(cname, n)] = (float(np.mean(gaps)), float(np.std(gaps)))
        print(f"  {cname:13s} N={n:3d}  gap={np.mean(gaps):+.3f} +- {np.std(gaps):.3f}")

print("\n=== mean gap across sizes (lower = better regularisation) ===")
for c in conds:
    mu = np.mean([results[(c,n)][0] for n in sizes]); print(f"  {c:13s}: {mu:+.3f}")

## 5. Figure

In [ ]:
import matplotlib.pyplot as plt
x = np.arange(len(sizes)); w = 0.26
cols = {"VQC (none)":"#8aa0b6","Depol 0.05":"#d1242f","QDropout 0.3":"#2da44e"}
fig, ax = plt.subplots(figsize=(7,4.2))
for k,c in enumerate(conds):
    mu = [results[(c,n)][0] for n in sizes]; sd = [results[(c,n)][1] for n in sizes]
    ax.bar(x+(k-1)*w, mu, w, yerr=sd, capsize=3, color=cols[c], edgecolor="white", label=c)
ax.set_xticks(x); ax.set_xticklabels([f"N={n}" for n in sizes])
ax.set_ylabel("Train-test accuracy gap"); ax.axhline(0, color="#666", lw=0.8)
ax.set_title("Per-gate quantum dropout regularises where depolarising noise fails")
ax.legend(frameon=False); plt.tight_layout(); plt.show()